In [1]:
# Structured Output
# Structured output is a type of output that is organized in a specific format, such as JSON, XML, or CSV. It allows for easy parsing and manipulation of the data by other programs or systems.
# Structured output is often used in APIs, data exchange, and logging systems to provide a consistent and machine-readable format for the output data. It can include key-value pairs, nested structures, and arrays to represent complex data relationships.
# For example, a structured output in JSON format might look like this:
{
  "name": "John Doe",
  "age": 30,
  "email": "john.doe@example.com"
}


# In this example, the output is organized as a JSON object with three key-value pairs: "name", "age", and "email". This structured format allows for easy access to the individual pieces of data and can be easily parsed by other programs or systems.


#We can use three ways to generate structured output in Python:
# 1. Using the Typeddict from the typing module to define a structured output type.
# 2. Using Pydantic models to define structured output with validation.
# 3. Using json.dumps() to convert a Python dictionary to a JSON string for structured output.

#Note: The following code snippets demonstrate each of these methods for generating structured output in Python.

{'name': 'John Doe', 'age': 30, 'email': 'john.doe@example.com'}

**Using TypedDict**

In [2]:
# 1. Using Typeddict from the typing module
from typing import TypedDict

class Person(TypedDict):
    name: str
    age: int

new_person: Person = {'name': 'John Doe', 'age': 30}

print(new_person)

{'name': 'John Doe', 'age': 30}


In [1]:
# 2. Using Pydantic models to define structured output with validation
from pydantic import BaseModel,EmailStr,Field
from typing import Optional

class Student(BaseModel):
    name: str = Field(description='The name of the student', min_length=1)
    age: int
    email: EmailStr
    grade: Optional[float] = Field(default=None, ge=0, le=100,description='The grade of the student, must be between 0 and 100')

new_student = Student(name='Jane Doe', age=22, email='jane.doe@example.com', grade=85.5)
print(new_student)

#or
new_student = {'name': 'Jane Doe', 'age': 22, 'email': 'jane.doe@example.com', 'grade': 85.5}
new_student_model = Student(**new_student)
print(new_student_model)

name='Jane Doe' age=22 email='jane.doe@example.com' grade=85.5
name='Jane Doe' age=22 email='jane.doe@example.com' grade=85.5


In [2]:
# 3. Using json.dumps() to convert a Python dictionary to a JSON string for structured output
import json
{
    "title": "student",
    "description": "schema about students",
    "type": "object",
    "properties":{
        "name":"string",
        "age":"integer"
    },
    "required":["name"]
}



{'title': 'student',
 'description': 'schema about students',
 'type': 'object',
 'properties': {'name': 'string', 'age': 'integer'},
 'required': ['name']}

**Application in LLM Models - UseCase**

In [3]:
#Few model providers, such as OpenAI, support structured output natively. This means that you can specify the desired output format directly in your API calls, and the model will return the output in that format. For example, you can specify that you want the output to be in JSON format, and the model will return a JSON object with the generated content. This can be particularly useful for applications that require structured data for further processing or integration with other systems.

#By using with_structured_output() method, you can specify the desired output format for the model's response. This allows you to receive the output in a structured format, such as JSON or XML, which can be easily parsed and utilized in your application. The method takes a parameter that defines the structure of the output, ensuring that the model's response adheres to the specified format. This is particularly useful when you need to integrate the model's output with other systems or when you want to ensure consistency in the data format for further processing.
#and some model providers, such as Hugging Face, do not support structured output natively. In such cases, you can still achieve structured output by post-processing the model's response. This involves taking the raw output from the model and transforming it into the desired structured format using additional code or libraries. For example, you can use regular expressions, string manipulation, or JSON parsing to extract relevant information from the model's response and organize it into a structured format that suits your needs. While this approach requires extra effort compared to native support, it allows you to work with models that do not have built-in structured output capabilities.
#those kind of require output parsing, which is a technique used to extract specific information from the raw output generated by a model. This is often necessary when the model does not support structured output natively, and you need to post-process the output to obtain the desired format. Output parsing can involve using regular expressions, string manipulation, or other techniques to identify and extract relevant data from the model's response. This allows you to transform the raw output into a structured format that can be easily utilized in your application or integrated with other systems.

In [4]:
# 1. Using Typeddict from the typing module
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Optional, Literal

In [5]:
load_dotenv()

True

In [10]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [13]:
#schema for structured output
class Review(TypedDict):
    key_themes: Annotated[list[str], "Write down all the key themes discuss in the review in a list"]
    summary: Annotated[str, "A brief summary of review"]
    sentiment: Annotated[Literal['pos','neg','neutral'],"Return sentiment of the review either negative, positive or neutral "]
    pros: Annotated[Optional[list[str]], "Write down all the pros inside a list"]
    cons: Annotated[Optional[list[str]], "Write down all the cons inside a list"]
    name: Annotated[Optional[str], "Write the name of the reviewer, please avoid to include any other text(like product name) except the name of the reviewer"]


review = '''
I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, it’s an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything lightning fast—whether I’m gaming, multitasking, or editing photos. The 5000mAh battery easily lasts a full day even with heavy use, and the 45W fast charging is a lifesaver.

The S-Pen integration is a great touch for note-taking and quick sketches, though I don't use it often. What really blew me away is the 200MP camera—the night mode is stunning, capturing crisp, vibrant images even in low light. Zooming up to 100x actually works well for distant objects, but anything beyond 30x loses quality.

However, the weight and size make it a bit uncomfortable for one-handed use. Also, Samsung’s One UI still comes with bloatware—why do I need five different Samsung apps for things Google already provides? The $1,300 price tag is also a hard pill to swallow.

Pros:
Insanely powerful processor (great for gaming and productivity)
Stunning 200MP camera with incredible zoom capabilities
Long battery life with fast charging
S-Pen support is unique and useful
                                 
Review by John Doe
'''


structured_model = model.with_structured_output(Review)
response = structured_model.invoke(review)

print(response)


{'key_themes': ['performance', 'camera quality', 'battery life', 'fast charging', 'S-Pen', 'device size and weight', 'bloatware', 'price'], 'summary': 'The Samsung Galaxy S24 Ultra is a high-performance smartphone with a powerful Snapdragon 8 Gen 3 processor, excellent 200MP camera with impressive night mode and zoom, and a long-lasting 5000mAh battery with 45W fast charging. The S-Pen is a useful addition. However, its large size and weight make one-handed use difficult, it comes with bloatware, and its $1,300 price tag is a significant drawback. Zoom quality also diminishes beyond 30x.', 'sentiment': 'pos', 'pros': ['Insanely powerful processor (great for gaming and productivity)', 'Stunning 200MP camera with incredible zoom capabilities', 'Long battery life with fast charging', 'S-Pen support is unique and useful'], 'cons': ['Weight and size make it uncomfortable for one-handed use', 'Samsung’s One UI comes with bloatware', 'High $1,300 price tag', 'Zoom quality loses quality beyond

In [14]:
# 2. Using Pydantic models to define structured output with validation

# Schema for structured output using Pydantic
#libraries and model setup already done in the first part

class ReviewModel(BaseModel):
    key_themes: list[str] =  Field(description="Write down all the key themes discuss in the review in a list")
    summary: str = Field(description="A brief summary of review")
    sentiment: Literal['pos','neg','neutral'] = Field(description="Return sentiment of the review either negative, positive or neutral")
    pros: Optional[list[str]] = Field(description="Write down all the pros inside a list")
    cons: Optional[list[str]] = Field(description="Write down all the cons inside a list")
    name: Optional[str] = Field(description="Write the name of the reviewer, please avoid to include any other text(like product name) except the name of the reviewer")

structured_model = model.with_structured_output(ReviewModel)

#review variable already defined in the first part
response = structured_model.invoke(review)

print(response)


key_themes=['performance', 'camera quality', 'battery life', 'S-Pen functionality', 'device size and weight', 'software experience', 'price'] summary='The Samsung Galaxy S24 Ultra is praised for its powerful Snapdragon 8 Gen 3 processor, long-lasting 5000mAh battery with 45W fast charging, and an impressive 200MP camera with excellent night mode and up to 30x usable zoom. The S-Pen integration is also noted as a useful feature. However, the phone is criticized for its heavy and large design, bloatware in One UI, and high price tag.' sentiment='pos' pros=['Insanely powerful processor (great for gaming and productivity)', 'Stunning 200MP camera with incredible zoom capabilities', 'Long battery life with fast charging', 'S-Pen support is unique and useful', 'Excellent night mode camera performance', 'Effective zoom up to 30x'] cons=['Heavy and large, making one-handed use uncomfortable', "Samsung's One UI includes bloatware", 'High price tag ($1,300)', 'Zoom beyond 30x loses quality'] nam

In [15]:
# 3. Using json.dumps() to convert a Python dictionary to a JSON string for structured output

# schema
json_schema = {
  "title": "Review",
  "type": "object",
  "properties": {
    "key_themes": {
      "type": "array",
      "items": {
        "type": "string"
      },
      "description": "Write down all the key themes discussed in the review in a list"
    },
    "summary": {
      "type": "string",
      "description": "A brief summary of the review"
    },
    "sentiment": {
      "type": "string",
      "enum": ["pos", "neg"],
      "description": "Return sentiment of the review either negative, positive or neutral"
    },
    "pros": {
      "type": ["array", "null"],
      "items": {
        "type": "string"
      },
      "description": "Write down all the pros inside a list"
    },
    "cons": {
      "type": ["array", "null"],
      "items": {
        "type": "string"
      },
      "description": "Write down all the cons inside a list"
    },
    "name": {
      "type": ["string", "null"],
      "description": "Write the name of the reviewer"
    }
  },
  "required": ["key_themes", "summary", "sentiment"]
}


structured_model = model.with_structured_output(json_schema)

#review variable already defined in the first part
response = structured_model.invoke(review)

print(response)

{'key_themes': ['Performance', 'Camera Quality', 'Battery Life', 'S-Pen Integration', 'Device Size and Weight', 'Software Bloatware', 'Price'], 'summary': "John Doe reviews the Samsung Galaxy S24 Ultra, praising its powerful Snapdragon 8 Gen 3 processor, long-lasting 5000mAh battery with fast charging, and exceptional 200MP camera, especially its night mode and zoom up to 100x. The S-Pen is also a valued feature. However, he notes its uncomfortable size and weight for one-handed use, the presence of Samsung's bloatware, and its high price point. He also mentions that zoom quality degrades beyond 30x.", 'sentiment': 'pos', 'pros': ['Insanely powerful processor (great for gaming and productivity)', 'Stunning 200MP camera with incredible zoom capabilities', 'Long battery life with fast charging', 'S-Pen support is unique and useful'], 'cons': ['Weight and size make it a bit uncomfortable for one-handed use', 'Samsung’s One UI still comes with bloatware', 'The $1,300 price tag is a hard pi

In [16]:
#In summary, structured output is a way to organize data in a specific format that can be easily parsed and utilized by other programs or systems. Some model providers support structured output natively, while others require post-processing and output parsing to achieve the desired format. By using techniques such as Typeddict, Pydantic models, or JSON conversion, you can generate structured output in Python and ensure that your model's responses are organized and consistent for further processing.

#Benefit of structured output:
#1. Improved Data Organization: Structured output allows for better organization of data, making it easier to access and manipulate specific pieces of information. This can lead to more efficient data processing and analysis.
#2. Enhanced Data Consistency: Structured output ensures that data is formatted in a consistent manner, reducing the likelihood of errors and making it easier to integrate with other systems.
#3. Better Integration: Structured output can be easily integrated with other systems, such as databases, APIs, or data analysis tools, allowing for seamless data exchange and processing.

#Key Difference between typeddict and pydantic and jsonschema, and benefit of each:
#1. Typeddict: Typeddict is a feature in Python's typing module that allows you to define a structured output type using a dictionary-like syntax. It provides a way to specify the expected keys and their corresponding types, making it easier to work with structured data in Python. The benefit of using Typeddict is that it is simple and lightweight, making it suitable for small projects or when you want to quickly define a structured output without additional dependencies.
#but typeddict does not provide built-in validation or error handling, so you need to ensure that the data adheres to the specified schema manually.
#2. Pydantic Models: Pydantic is a powerful library that allows you to define structured output using Python classes. It provides built-in validation and parsing capabilities, making it easier to ensure that the data adheres to the specified schema. The benefit of using Pydantic models is that it offers robust validation and error handling, which can help catch issues early in the development process.
#but Pydantic models can be more complex and may require additional dependencies, making it less suitable for small projects or when you want a lightweight solution.
#also pydanctic models are not easy in intrigation with other programming languages, as they are specific to Python. which pushes the need of json schema, which is a language-agnostic way to define structured output.
#3. JSON Schema: JSON Schema is a standard for defining the structure of JSON data. It allows you to specify the expected format, types, and constraints for JSON data. The benefit of using JSON Schema is that it is widely supported and can be used across different programming languages and platforms. It provides a standardized way to validate and document JSON data, making it easier to ensure consistency and interoperability in your applications.
